# SMFRD Extension: Synthetic Train, Real Eval

This notebook is an optional extension to the final pair-head project. It prepares datasets in a clean Colab runtime, trains the pair-head adapter on a larger synthetic masked/unmasked root, and evaluates on real RMFRD masked/unmasked identities.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/tungd/masked-face-id.git"
REPO_DIR = Path("/content/masked-face-id")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%cd /content/masked-face-id
!git pull --ff-only origin main
!python scripts/install_colab_deps.py

## Dataset Setup

Start by summarizing `/content/datasets`. On a fresh reconnected runtime this should be empty. Then choose one of the download paths below.

RMFRD is the real-mask evaluation root. SMFRD or LFW-SMFRD is the synthetic training source. The pair-head training root must contain both `masked/<identity>` and `unmasked/<identity>` images, so a masked-only SMFRD mirror must be paired with the original unmasked source images.

In [ ]:
!python scripts/setup_masked_datasets.py --root /content/datasets --summarize-only

### RMFRD Real-Mask Evaluation Data

Use either the official Google Drive link or the Kaggle mirror. The Kaggle path requires `kaggle.json` credentials in the runtime.

In [ ]:
# Official Google Drive path from the dataset README.
# If this fails because Google Drive quota changes, use the Kaggle cell below.
!python scripts/setup_masked_datasets.py --root /content/datasets --rmfrd-source gdrive

In [ ]:
# Optional Kaggle fallback. Requires /root/.kaggle/kaggle.json.
# !python -m pip install -q kaggle
# !python scripts/setup_masked_datasets.py --root /content/datasets --rmfrd-source kaggle

### SMFRD / LFW-SMFRD Training Data

The Kaggle LFW-SMFRD mirror contains simulated masked LFW images. Pair it with original unmasked LFW images before training. If you already mounted an official SMFRD archive from Drive, use `--smfrd-source archive` or skip download and point `--masked-root` / `--unmasked-root` to the extracted folders.

In [ ]:
# Optional Kaggle mirror. Requires /root/.kaggle/kaggle.json.
# !python -m pip install -q kaggle
# !python scripts/setup_masked_datasets.py --root /content/datasets --smfrd-source kaggle

In [ ]:
# Edit these two roots after downloading or mounting the masked and unmasked sources.
SMFRD_MASKED_ROOT = Path("/content/datasets/smfrd/kaggle/lfw_masked/lfw_train")
SMFRD_UNMASKED_ROOT = Path("/content/datasets/lfw/original")
TRAIN_ROOT = Path("/content/datasets/normalized/lfw_smfrd_paired")

if SMFRD_MASKED_ROOT.exists() and SMFRD_UNMASKED_ROOT.exists():
    !python scripts/setup_masked_datasets.py \
      --root /content/datasets \
      --make-paired-view \
      --masked-root {SMFRD_MASKED_ROOT} \
      --unmasked-root {SMFRD_UNMASKED_ROOT} \
      --paired-view-root {TRAIN_ROOT}
else:
    print("Edit SMFRD_MASKED_ROOT and SMFRD_UNMASKED_ROOT, then rerun this cell.")

### Reproducible Fallback

If Kaggle or the official SMFRD archives are unavailable in the clean runtime, create a deterministic LFW masked/unmasked paired root. This is not the official SMFRD release, but it keeps the larger synthetic-train, real-eval probe reproducible.

In [ ]:
!python scripts/create_lfw_synthetic_mask_pairs.py \
  --out-root /content/datasets/normalized/lfw_synthetic_mask_pairs \
  --min-faces-per-person 2 \
  --max-identities 1200 \
  --max-images-per-identity 8 \
  --seed 42

In [ ]:
# Update these after setup_masked_datasets.py prints candidate roots.
TRAIN_ROOT = Path("/content/datasets/normalized/lfw_synthetic_mask_pairs")
EVAL_ROOT = Path("/content/datasets/rmfrd/extracted/self-built-masked-face-recognition-dataset")
OUT_DIR = Path("/content/masked_face_final_runs/lfw_synth_train_rmfrd_eval_seed42")

print({"train_root_exists": TRAIN_ROOT.exists(), "eval_root_exists": EVAL_ROOT.exists(), "out_dir": str(OUT_DIR)})

## Run Probe

This is the expensive GPU step. It trains on `TRAIN_ROOT` and evaluates on `EVAL_ROOT`.

In [ ]:
if not TRAIN_ROOT.exists():
    raise FileNotFoundError(f"Missing TRAIN_ROOT: {TRAIN_ROOT}")
if not EVAL_ROOT.exists():
    raise FileNotFoundError(f"Missing EVAL_ROOT: {EVAL_ROOT}")

!python scripts/probe_pair_head_synthetic_train_real_eval.py \
  --train-data-root {TRAIN_ROOT} \
  --eval-data-root {EVAL_ROOT} \
  --out-dir {OUT_DIR} \
  --train-identities 1000 \
  --eval-identities 80 \
  --max-train-images-per-condition 4 \
  --max-eval-images-per-condition 8 \
  --train-pairs-per-case 6000 \
  --eval-pairs-per-case 800 \
  --epochs 50 \
  --dropout 0.30 \
  --seed 42

In [ ]:
import pandas as pd

results = pd.read_csv(OUT_DIR / "synthetic_train_real_eval_results.csv")
display(results)
print((OUT_DIR / "synthetic_train_real_eval_conclusion.md").read_text())